# (Optional) Setup AgentCore Gateway (legacy)

`02_setup_agentcore_gateway.ipynb`에서 설정한 Cognito 인증을 재사용하여
`fhir-mcp-legacy` Lambda용 AgentCore Gateway를 추가합니다.

## Step 1: Configuration

In [ ]:
import boto3, json, time, os

REGION = boto3.session.Session().region_name or os.environ.get("AWS_DEFAULT_REGION") or os.environ.get("AWS_REGION")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

iam_client = boto3.client("iam")
cognito_client = boto3.client("cognito-idp", region_name=REGION)
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)

# Load existing Cognito info from notebook 02
with open("connection_info.json") as f:
    main_conn = json.load(f)

USER_POOL_ID = main_conn["user_pool_id"]
CLIENT_ID = main_conn["client_id"]
CLIENT_SECRET = main_conn["client_secret"]
TOKEN_URL = main_conn["token_url"]
DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"

FUNC_NAME = "fhir-mcp-legacy"
GATEWAY_NAME = "fhir-mcp-legacy-gw"
GATEWAY_ROLE = "fhir-mcp-legacy-gw-role"
TARGET_NAME = "LegacyTarget"

print(f"Account: {ACCOUNT_ID}, Region: {REGION}")
print(f"Reusing Cognito User Pool: {USER_POOL_ID}")


## Step 2: Create Gateway IAM Role

In [ ]:
lambda_arn = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{FUNC_NAME}"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "bedrock-agentcore.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": "lambda:InvokeFunction", "Resource": lambda_arn}]
}

try:
    iam_client.create_role(RoleName=GATEWAY_ROLE, AssumeRolePolicyDocument=json.dumps(trust_policy))
    print(f"Created role: {GATEWAY_ROLE}")
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"Role exists: {GATEWAY_ROLE}")

iam_client.put_role_policy(RoleName=GATEWAY_ROLE, PolicyName="InvokeLambda", PolicyDocument=json.dumps(inline_policy))
GATEWAY_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{GATEWAY_ROLE}"
print(f"Role ARN: {GATEWAY_ROLE_ARN}")
time.sleep(10)


## Step 3: Create Gateway & Target

In [ ]:
TOOL_SCHEMAS = [{"name": "list_tables", "description": "List available FHIR data lake tables.", "inputSchema": {"type": "object", "properties": {"domain": {"type": "string"}}}}, {"name": "get_table_schema", "description": "Get table schema with column names and types.", "inputSchema": {"type": "object", "properties": {"table_name": {"type": "string"}}, "required": ["table_name"]}}, {"name": "get_table_relationships", "description": "Get table relationships for JOIN queries.", "inputSchema": {"type": "object", "properties": {"table_name": {"type": "string"}}}}, {"name": "run_custom_query", "description": "Execute a read-only Spark SQL SELECT query. LIMIT 100 enforced.", "inputSchema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}]

gateway = agentcore_client.create_gateway(
    name=GATEWAY_NAME, roleArn=GATEWAY_ROLE_ARN, protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {"discoveryUrl": DISCOVERY_URL, "allowedClients": [CLIENT_ID]}}
)
GATEWAY_ID = gateway["gatewayId"]
GATEWAY_URL = gateway["gatewayUrl"]
print(f"Gateway ID: {GATEWAY_ID}")
print(f"Gateway URL: {GATEWAY_URL}")

print("Waiting for gateway READY...")
for _ in range(30):
    status = agentcore_client.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
    if status == "READY":
        print("Gateway is READY!")
        break
    time.sleep(5)

try:
    lambda_client.add_permission(
        FunctionName=FUNC_NAME, StatementId="AgentCoreGateway-legacy",
        Action="lambda:InvokeFunction", Principal="bedrock-agentcore.amazonaws.com",
        SourceArn=f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{GATEWAY_ID}"
    )
    print("Lambda permission added")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists")

target = agentcore_client.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID, name=TARGET_NAME,
    targetConfiguration={"mcp": {"lambda": {"lambdaArn": lambda_arn, "toolSchema": {"inlinePayload": TOOL_SCHEMAS}}}},
    credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}]
)
TARGET_ID = target["targetId"]
print(f"Target ID: {TARGET_ID}")


## Step 4: Save Connection Info

In [ ]:
legacy_conn = {
    "legacy": {
        "gateway_id": GATEWAY_ID,
        "gateway_url": GATEWAY_URL,
        "target_id": TARGET_ID,
        "target_name": TARGET_NAME,
    },
    "auth": {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "token_url": TOKEN_URL,
        "user_pool_id": USER_POOL_ID,
    }
}

with open("legacy_connection_info.json", "w") as f:
    json.dump(legacy_conn, f, indent=2)
print("Saved to legacy_connection_info.json")
print(f"\nGateway URL: {GATEWAY_URL}")
print(f"Client ID  : {CLIENT_ID}")
print(f"Token URL  : {TOKEN_URL}")


## ✅ Done!

`fhir-mcp-legacy-gw` Gateway가 설정되었습니다.

다음: `08_deploy_legacy_agent.ipynb`에서 AgentCore Runtime을 배포합니다.